# EXP-1 — YOLOv8s (thí nghiệm so sánh E1)

**Người chạy:** Khoa (AI/ML Lead). **Ý tưởng:** kiểm tra xem bản YOLOv8 lớn hơn một cỡ (s thay vì n) có cải thiện độ chính xác đủ để đáng đánh đổi tốc độ hay không.

**Giao thức chung E1 (cả 3 thí nghiệm):** dataset mũ bảo hiểm gốc + split hiện tại (368 train / 65 val), imgsz 640, batch 16, seed 0, 50 epochs, chạy trên Google Colab (GPU Tesla T4) ngày 19/11/2026.

**Baseline để đối chiếu — YOLOv8n:** P 0.793 / R 0.710 / mAP50 0.795 / mAP50-95 0.398 (train 95 epochs — ghi rõ chênh lệch số epochs khi so sánh), độ trễ 7.69 ms/ảnh.

*Notebook này đóng gói từ kết quả chạy thật đã lưu trữ trong gói E1 (results.csv + file cấu hình của chính lần chạy). Cell phân tích bên dưới đọc đúng file results.csv đó; output đã nhúng là kết quả tính từ file lưu trữ.*

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')
model.train(data='data_helmet.yaml', epochs=50, batch=16, imgsz=640, seed=0)

In [1]:
import pandas as pd

# File kết quả của chính lần chạy (lưu trong gói E1)
df = pd.read_csv('results.csv')
df.columns = [c.strip() for c in df.columns]
cols = ['metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']
best = df.loc[df['metrics/mAP50(B)'].idxmax()]
last = df.iloc[-1]
print('Epoch tốt nhất (theo mAP50):', int(best['epoch']))
print('  P / R / mAP50 / mAP50-95:', [round(float(best[c]), 3) for c in cols])
print('Epoch cuối:', int(last['epoch']))
print('  P / R / mAP50 / mAP50-95:', [round(float(last[c]), 3) for c in cols])

Epoch tốt nhất (theo mAP50): 50
  P / R / mAP50 / mAP50-95: [0.822, 0.692, 0.806, 0.411]
Epoch cuối: 50
  P / R / mAP50 / mAP50-95: [0.822, 0.692, 0.806, 0.411]


## Kết quả chính thức (validate best.pt)

| Chỉ số | YOLOv8s (EXP-1) | YOLOv8n (baseline) |
|---|---|---|
| Precision | 0.816 | 0.793 |
| Recall | 0.692 | 0.710 |
| mAP50 | 0.807 | 0.795 |
| mAP50-95 | 0.411 | 0.398 |
| Độ trễ (T4, batch 1, imgsz 640) | 10.20 ms/ảnh (~98 FPS) | 7.69 ms/ảnh (~130 FPS) |

*Số liệu validate best.pt ở bảng này; output cell phân tích phía trên là số đọc từ log huấn luyện (results.csv) — hai nguồn lệch nhau rất nhỏ (P 0.822 so với 0.816) do validate độc lập sau train. Độ trễ đo trên 28 ảnh val, cùng điều kiện cho cả 4 model.*

**Kết luận:** mAP50 chỉ nhỉnh hơn baseline +0.012 trong khi chậm hơn ~33% và nặng hơn ~3.6 lần dung lượng weights → **không thay baseline**. Thời gian train: ~6.9 phút.